In [4]:
from pathlib import Path
import sys

sys.path.append(str(Path("..").resolve()))

from src.tse_io import *
from src.graph_builder import *
import networkx as nx

In [7]:
zip_path = Path("C:\\Users\\rafael.watanabe\\git\\trabalho_conclusao_t1_ifgo\\data\\raw\\prestacao_de_contas_eleitorais_candidatos_2022.zip")
files = list_zip_files(zip_path)
files[:10]

['despesas_contratadas_candidatos_2022_BA.csv',
 'despesas_contratadas_candidatos_2022_AP.csv',
 'receitas_candidatos_2022_BRASIL.csv',
 'despesas_contratadas_candidatos_2022_AL.csv',
 'despesas_contratadas_candidatos_2022_AM.csv',
 'despesas_contratadas_candidatos_2022_AC.csv',
 'despesas_contratadas_candidatos_2022_CE.csv',
 'despesas_pagas_candidatos_2022_BA.csv',
 'despesas_contratadas_candidatos_2022_ES.csv',
 'despesas_contratadas_candidatos_2022_GO.csv']

In [8]:
receitas = read_brasil_file(
    zip_path,
    "receitas_candidatos_2022"
)

despesas_contratadas = read_brasil_file(
    zip_path,
    "despesas_contratadas_candidatos_2022"
)

receitas_originario = read_brasil_file(
    zip_path,
    "receitas_candidatos_doador_originario_2022"
)

In [9]:
receitas_presidente = filter_by_cargo(receitas, "Presidente")

In [10]:
receitas_pres = filter_presidential_candidates(receitas)
despesas_pres = filter_presidential_candidates(despesas_contratadas)

In [9]:
receitas_pres[["NR_CANDIDATO", "NM_CANDIDATO", "SG_PARTIDO"]].drop_duplicates()

,NR_CANDIDATO,NM_CANDIDATO,SG_PARTIDO
1534,22,JAIR MESSIAS BOLSONARO,PL
21188,13,LUIZ INÁCIO LULA DA SILVA,PT


In [10]:
despesas_pres[["NR_CANDIDATO", "NM_CANDIDATO", "SG_PARTIDO"]].drop_duplicates()

,NR_CANDIDATO,NM_CANDIDATO,SG_PARTIDO
558517,13,LUIZ INÁCIO LULA DA SILVA,PT
563482,22,JAIR MESSIAS BOLSONARO,PL


In [11]:
sq_receitas_pres = receitas_pres["SQ_RECEITA"].dropna().unique()

receitas_originario_pres = receitas_originario[
    receitas_originario["SQ_RECEITA"].isin(sq_receitas_pres)
].copy()

In [12]:
receitas_com_originario = receitas_pres.merge(
    receitas_originario_pres,
    on="SQ_RECEITA",
    how="left",
    suffixes=("", "_ORIGINARIO")
)

In [13]:
receitas_com_originario[
    [
        "SQ_RECEITA",
        "NM_CANDIDATO",
        "SG_PARTIDO",
        "DS_FONTE_RECEITA",
        "DS_ORIGEM_RECEITA",
        "NM_DOADOR",
        "NM_DOADOR_ORIGINARIO",
        "DS_CNAE_DOADOR_ORIGINARIO",
        "VR_RECEITA",
    ]
].head()

,SQ_RECEITA,NM_CANDIDATO,SG_PARTIDO,DS_FONTE_RECEITA,DS_ORIGEM_RECEITA,NM_DOADOR,NM_DOADOR_ORIGINARIO,DS_CNAE_DOADOR_ORIGINARIO,VR_RECEITA
0,46245435,JAIR MESSIAS BOLSONARO,PL,OUTROS RECURSOS,Recursos de partido político,DIREÇÃO NACIONAL,NaN,NaN,"2412,88"
1,46245435,JAIR MESSIAS BOLSONARO,PL,OUTROS RECURSOS,Recursos de partido político,DIREÇÃO NACIONAL,NaN,NaN,"1532,45"
2,46245435,JAIR MESSIAS BOLSONARO,PL,OUTROS RECURSOS,Recursos de partido político,DIREÇÃO NACIONAL,NaN,NaN,"1532,45"
3,46245435,JAIR MESSIAS BOLSONARO,PL,OUTROS RECURSOS,Recursos de partido político,DIREÇÃO NACIONAL,NaN,NaN,"1718,87"
4,46245435,JAIR MESSIAS BOLSONARO,PL,OUTROS RECURSOS,Recursos de partido político,DIREÇÃO NACIONAL,NaN,NaN,"827,87"


In [14]:
receitas_com_originario["VR_RECEITA_NUM"] = parse_money(
    receitas_com_originario["VR_RECEITA"]
)

despesas_pres["VR_DESPESA_CONTRATADA_NUM"] = parse_money(
    despesas_pres["VR_DESPESA_CONTRATADA"]
)

In [15]:
(
    receitas_pres
    .assign(VR_RECEITA_NUM=parse_money(receitas_pres["VR_RECEITA"]))
    .groupby(["NR_CANDIDATO", "NM_CANDIDATO", "SG_PARTIDO"], as_index=False)
    .agg(
        total_receita=("VR_RECEITA_NUM", "sum"),
        qtd_receitas=("SQ_RECEITA", "nunique")
    )
    .sort_values("total_receita", ascending=False)
)

,NR_CANDIDATO,NM_CANDIDATO,SG_PARTIDO,total_receita,qtd_receitas
0,13,LUIZ INÁCIO LULA DA SILVA,PT,1.355393e+08,382
1,22,JAIR MESSIAS BOLSONARO,PL,1.261275e+08,392012


In [16]:
(
    despesas_pres
    .assign(VR_DESPESA_CONTRATADA_NUM=parse_money(despesas_pres["VR_DESPESA_CONTRATADA"]))
    .groupby(["NR_CANDIDATO", "NM_CANDIDATO", "SG_PARTIDO"], as_index=False)
    .agg(
        total_despesa_contratada=("VR_DESPESA_CONTRATADA_NUM", "sum"),
        qtd_despesas=("SQ_DESPESA", "nunique")
    )
    .sort_values("total_despesa_contratada", ascending=False)
)

,NR_CANDIDATO,NM_CANDIDATO,SG_PARTIDO,total_despesa_contratada,qtd_despesas
0,13,LUIZ INÁCIO LULA DA SILVA,PT,1.313130e+08,963
1,22,JAIR MESSIAS BOLSONARO,PL,8.906076e+07,672


In [17]:
(
    receitas_pres
    .assign(VR_RECEITA_NUM=parse_money(receitas_pres["VR_RECEITA"]))
    .groupby(["NM_CANDIDATO", "NM_DOADOR", "DS_CNAE_DOADOR"], as_index=False)
    .agg(total_receita=("VR_RECEITA_NUM", "sum"))
    .sort_values("total_receita", ascending=False)
    .head(20)
)

,NM_CANDIDATO,NM_DOADOR,DS_CNAE_DOADOR,total_receita
361202,LUIZ INÁCIO LULA DA SILVA,Direção Nacional,Atividades de organizações políticas,1.243279e+08
81017,JAIR MESSIAS BOLSONARO,DIREÇÃO NACIONAL,Atividades de organizações políticas,3.669334e+07
361311,LUIZ INÁCIO LULA DA SILVA,UM A MAIS SERVIÇOS DE TECNOLOGIA E CONSULTORIA,"Suporte técnico, manutenção e outros serviços ...",6.814847e+06
109208,JAIR MESSIAS BOLSONARO,FABIANO CAMPOS ZETTEL,#NULO,3.000000e+06
189767,JAIR MESSIAS BOLSONARO,JOSÉ SALIM MATTAR JUNIOR,#NULO,1.800000e+06
145527,JAIR MESSIAS BOLSONARO,HUGO DE CARVALHO RIBEIRO,#NULO,1.200000e+06
361200,LUIZ INÁCIO LULA DA SILVA,Direção Estadual/Distrital,Atividades de organizações políticas,1.166137e+06
212504,JAIR MESSIAS BOLSONARO,LUCIANO HANG,#NULO,1.022000e+06
284091,JAIR MESSIAS BOLSONARO,PEDRO GRENDENE BARTELLE,#NULO,1.000000e+06
273621,JAIR MESSIAS BOLSONARO,OSCAR LUIZ CERVI,#NULO,1.000000e+06


In [18]:
(
    receitas_com_originario
    .groupby(
        [
            "NM_CANDIDATO",
            "NM_DOADOR_ORIGINARIO",
            "TP_DOADOR_ORIGINARIO",
            "DS_CNAE_DOADOR_ORIGINARIO",
        ],
        as_index=False,
    )
    .agg(total_receita=("VR_RECEITA_NUM", "sum"))
    .sort_values("total_receita", ascending=False)
    .head(20)
)

,NM_CANDIDATO,NM_DOADOR_ORIGINARIO,TP_DOADOR_ORIGINARIO,DS_CNAE_DOADOR_ORIGINARIO,total_receita
35317,LUIZ INÁCIO LULA DA SILVA,JOSE CARLOS PINHEIRO PRIOSTE,F,#NULO,9182279.41
3839,JAIR MESSIAS BOLSONARO,MAURO FIDELIX DA SILVA,F,#NULO,4791932.34
54399,LUIZ INÁCIO LULA DA SILVA,PAULO AVELINO BARBOSA SILVA,F,#NULO,4239326.76
51111,LUIZ INÁCIO LULA DA SILVA,MIGUEL LEONEL DOS SANTOS,F,#NULO,3403540.38
28465,LUIZ INÁCIO LULA DA SILVA,GILBERTO MAIA,F,#NULO,3150083.17
3735,JAIR MESSIAS BOLSONARO,MARIO ALBERTO MARCHINI,F,#NULO,3064476.91
63986,LUIZ INÁCIO LULA DA SILVA,TARCISIO PRACIANO PEREIRA,F,#NULO,3040936.12
26063,LUIZ INÁCIO LULA DA SILVA,FERNANDO MORENO,F,#NULO,2954110.74
47619,LUIZ INÁCIO LULA DA SILVA,MARIA ISABEL COSTA DA SILVEIRA,F,#NULO,2952258.82
5047,JAIR MESSIAS BOLSONARO,THIAGO VINICIUS TEIXEIRA PEREIRA,F,#NULO,2738564.05


In [19]:
(
    despesas_pres
    .assign(VR_DESPESA_CONTRATADA_NUM=parse_money(despesas_pres["VR_DESPESA_CONTRATADA"]))
    .groupby(
        ["NM_CANDIDATO", "NM_FORNECEDOR", "DS_CNAE_FORNECEDOR", "DS_ORIGEM_DESPESA"],
        as_index=False,
    )
    .agg(total_despesa=("VR_DESPESA_CONTRATADA_NUM", "sum"))
    .sort_values("total_despesa", ascending=False)
    .head(20)
)

,NM_CANDIDATO,NM_FORNECEDOR,DS_CNAE_FORNECEDOR,DS_ORIGEM_DESPESA,total_despesa
533,LUIZ INÁCIO LULA DA SILVA,M4 COMUNICAÇÃO E PROPAGANDA LTDA,Agências de publicidade,"Produção de programas de rádio, televisão ou v...",37899000.00
148,JAIR MESSIAS BOLSONARO,GOOGLE BRASIL INTERNET LTDA.,"Portais, provedores de conteúdo e outros servi...",Despesa com Impulsionamento de Conteúdos,27828538.79
473,LUIZ INÁCIO LULA DA SILVA,GOOGLE BRASIL INTERNET LTDA,"Portais, provedores de conteúdo e outros servi...",Despesa com Impulsionamento de Conteúdos,22933534.93
339,JAIR MESSIAS BOLSONARO,RM FILMES E PUBLICIDADE LTDA,"Atividades de produção cinematográfica, de víd...","Produção de programas de rádio, televisão ou v...",10389869.09
101,JAIR MESSIAS BOLSONARO,DIREÇÃO NACIONAL,Atividades de organizações políticas,Doações financeiras a outros candidatos/partidos,5469206.92
123,JAIR MESSIAS BOLSONARO,FACEBOOK SERVICOS ONLINE DO BRASIL LTDA.,"Agenciamento de espaços para publicidade, exce...",Despesa com Impulsionamento de Conteúdos,5182800.00
373,JAIR MESSIAS BOLSONARO,UNIPAUTA FORMULARIOS LTDA,Impressão de material de segurança,Publicidade por materiais impressos,4974100.00
535,LUIZ INÁCIO LULA DA SILVA,MAGNI E AR PRODUCOES E SHOWS LTDA,"Artes cênicas, espetáculos e atividades comple...",Eventos de promoção da candidatura,4915657.00
446,LUIZ INÁCIO LULA DA SILVA,Direção Estadual/Distrital,Atividades de organizações políticas,Doações financeiras a outros candidatos/partidos,4765500.00
255,JAIR MESSIAS BOLSONARO,MAGIC BEANS COMUNICACAO LTDA,Agências de publicidade,Serviços prestados por terceiros,4419900.00


In [20]:
(
    receitas_pres
    .assign(VR_RECEITA_NUM=parse_money(receitas_pres["VR_RECEITA"]))
    .groupby(
        ["NM_CANDIDATO", "DS_FONTE_RECEITA"],
        as_index=False
    )
    .agg(total=("VR_RECEITA_NUM", "sum"))
    .sort_values("total", ascending=False)
)

,NM_CANDIDATO,DS_FONTE_RECEITA,total
3,LUIZ INÁCIO LULA DA SILVA,FUNDO ESPECIAL,1.251190e+08
2,JAIR MESSIAS BOLSONARO,OUTROS RECURSOS,9.469116e+07
1,JAIR MESSIAS BOLSONARO,FUNDO PARTIDARIO,2.944739e+07
5,LUIZ INÁCIO LULA DA SILVA,OUTROS RECURSOS,1.017819e+07
0,JAIR MESSIAS BOLSONARO,FUNDO ESPECIAL,1.988973e+06
4,LUIZ INÁCIO LULA DA SILVA,FUNDO PARTIDARIO,2.420922e+05


In [21]:
(
    receitas_pres
    .assign(VR_RECEITA_NUM=parse_money(receitas_pres["VR_RECEITA"]))
    .groupby(
        ["NM_CANDIDATO", "DS_ORIGEM_RECEITA"],
        as_index=False
    )
    .agg(total=("VR_RECEITA_NUM", "sum"))
    .sort_values("total", ascending=False)
)

,NM_CANDIDATO,DS_ORIGEM_RECEITA,total
6,LUIZ INÁCIO LULA DA SILVA,Recursos de partido político,1.258297e+08
3,JAIR MESSIAS BOLSONARO,Recursos de pessoas físicas,8.815575e+07
2,JAIR MESSIAS BOLSONARO,Recursos de partido político,3.710355e+07
5,LUIZ INÁCIO LULA DA SILVA,Recursos de Financiamento Coletivo,6.814847e+06
7,LUIZ INÁCIO LULA DA SILVA,Recursos de pessoas físicas,2.829703e+06
0,JAIR MESSIAS BOLSONARO,Recursos de Financiamento Coletivo,5.361398e+05
4,JAIR MESSIAS BOLSONARO,Rendimentos de aplicações financeiras,3.280258e+05
8,LUIZ INÁCIO LULA DA SILVA,Rendimentos de aplicações financeiras,6.500255e+04
1,JAIR MESSIAS BOLSONARO,Recursos de outros candidatos,4.060000e+03


In [15]:
receitas_pres["VR_RECEITA_NUM"] = parse_money(receitas_pres["VR_RECEITA"])
despesas_pres["VR_DESPESA_CONTRATADA_NUM"] = parse_money(
    despesas_pres["VR_DESPESA_CONTRATADA"]
)

In [16]:
G_cat, edges_cat = build_category_graph(
    receitas_pres=receitas_pres,
    despesas_pres=despesas_pres,
)

G_cat.number_of_nodes(), G_cat.number_of_edges()

(144, 224)

In [11]:
edges_cat.sort_values("valor_total", ascending=False).head(20)

,source,target,tipo,valor_total,qtd_registros,weight
86,tipo_despesa::Despesa com Impulsionamento de C...,"cnae_fornecedor::Portais, provedores de conteú...",tipo_despesa_cnae_fornecedor,51149073.72,11,0.053333
41,candidato::LUIZ INÁCIO LULA DA SILVA,"tipo_despesa::Produção de programas de rádio, ...",candidato_tipo_despesa,37899000.00,2,0.054199
140,"tipo_despesa::Produção de programas de rádio, ...",cnae_fornecedor::Agências de publicidade,tipo_despesa_cnae_fornecedor,37899000.00,2,0.054199
6,candidato::JAIR MESSIAS BOLSONARO,tipo_despesa::Despesa com Impulsionamento de C...,candidato_tipo_despesa,33011338.79,28,0.054608
28,candidato::LUIZ INÁCIO LULA DA SILVA,tipo_despesa::Despesa com Impulsionamento de C...,candidato_tipo_despesa,25189936.88,38,0.055426
22,candidato::JAIR MESSIAS BOLSONARO,tipo_despesa::Serviços prestados por terceiros,candidato_tipo_despesa,16467955.79,133,0.056764
42,candidato::LUIZ INÁCIO LULA DA SILVA,tipo_despesa::Publicidade por adesivos,candidato_tipo_despesa,13228772.20,325,0.057478
97,tipo_despesa::Doações financeiras a outros can...,cnae_fornecedor::Atividades de organizações po...,tipo_despesa_cnae_fornecedor,13049706.92,89,0.057523
43,candidato::LUIZ INÁCIO LULA DA SILVA,tipo_despesa::Publicidade por materiais impressos,candidato_tipo_despesa,12892228.38,276,0.057563
21,candidato::JAIR MESSIAS BOLSONARO,tipo_despesa::Publicidade por materiais impressos,candidato_tipo_despesa,12598786.00,160,0.057640


In [14]:
[n for n in G_cat.nodes if "Recursos de pessoas físicas" in n]

['origem_receita::Recursos de pessoas físicas']

In [15]:
[n for n in G_cat.nodes if "Portais" in n]

['cnae_fornecedor::Portais, provedores de conteúdo e outros serviços de informação na Internet']

In [16]:
[n for n in G_cat.nodes if "GOOGLE" in n]

[]

In [17]:
from collections import Counter

Counter(n.split("::")[0] for n in G_cat.nodes)

Counter({'cnae_fornecedor': 96,
         'tipo_despesa': 32,
         'origem_receita': 5,
         'cnae_doador': 4,
         'fonte_receita': 3,
         'candidato': 2,
         'partido': 2})

In [18]:
sorted([n for n in G_cat.nodes if n.startswith("cnae_fornecedor::")])

['cnae_fornecedor::Acabamentos em fios, tecidos e artefatos têxteis',
 'cnae_fornecedor::Agenciamento de espaços para publicidade, exceto em veículos de comunicação',
 'cnae_fornecedor::Agências de notícias',
 'cnae_fornecedor::Agências de publicidade',
 'cnae_fornecedor::Agências de viagens',
 'cnae_fornecedor::Aluguel de máquinas e equipamentos não especificados anteriormente',
 'cnae_fornecedor::Aluguel de máquinas e equipamentos para construção sem operador',
 'cnae_fornecedor::Aluguel de objetos pessoais e domésticos não especificados anteriormente',
 'cnae_fornecedor::Artes cênicas, espetáculos e atividades complementares',
 'cnae_fornecedor::Atividades de Correio',
 'cnae_fornecedor::Atividades de associações de defesa de direitos sociais',
 'cnae_fornecedor::Atividades de consultoria em gestão empresarial',
 'cnae_fornecedor::Atividades de contabilidade, consultoria e auditoria contábil e tributária',
 'cnae_fornecedor::Atividades de ensino não especificadas anteriormente',
 'c

In [23]:
source = [n for n in G_cat.nodes if "Recursos de pessoas físicas" in n][0]
target = [n for n in G_cat.nodes if "Portais" in n][0]

path = nx.shortest_path(
    G_cat,
    source=source,
    target=target,
    weight="weight",
)

path

['origem_receita::Recursos de pessoas físicas',
 'fonte_receita::OUTROS RECURSOS',
 'partido::PL',
 'candidato::JAIR MESSIAS BOLSONARO',
 'tipo_despesa::Despesa com Impulsionamento de Conteúdos',
 'cnae_fornecedor::Portais, provedores de conteúdo e outros serviços de informação na Internet']

In [24]:
for i in range(len(path) - 1):
    u = path[i]
    v = path[i + 1]
    data = G_cat[u][v]

    print(f"{u} -> {v}")
    print(f"  tipo: {data['tipo']}")
    print(f"  valor_total: {data['valor_total']}")
    print(f"  qtd_registros: {data['qtd_registros']}")
    print(f"  custo: {data['weight']}")
    print()

origem_receita::Recursos de pessoas físicas -> fonte_receita::OUTROS RECURSOS
  tipo: origem_fonte
  valor_total: 90985448.51
  qtd_registros: 391906
  custo: 0.05174320220662043

fonte_receita::OUTROS RECURSOS -> partido::PL
  tipo: fonte_partido
  valor_total: 94691156.01
  qtd_registros: 391961
  custo: 0.05163653963579093

partido::PL -> candidato::JAIR MESSIAS BOLSONARO
  tipo: partido_candidato
  valor_total: 126127517.46
  qtd_registros: 392140
  custo: 0.050883324322359216

candidato::JAIR MESSIAS BOLSONARO -> tipo_despesa::Despesa com Impulsionamento de Conteúdos
  tipo: candidato_tipo_despesa
  valor_total: 33011338.79
  qtd_registros: 28
  custo: 0.054607920971987224

tipo_despesa::Despesa com Impulsionamento de Conteúdos -> cnae_fornecedor::Portais, provedores de conteúdo e outros serviços de informação na Internet
  tipo: tipo_despesa_cnae_fornecedor
  valor_total: 51149073.72
  qtd_registros: 11
  custo: 0.05333260812176827



In [25]:
origens_receita = sorted([
    n for n in G_cat.nodes
    if n.startswith("origem_receita::")
])

destinos_cnae = sorted([
    n for n in G_cat.nodes
    if n.startswith("cnae_fornecedor::")
])

origens_receita

['origem_receita::Recursos de Financiamento Coletivo',
 'origem_receita::Recursos de outros candidatos',
 'origem_receita::Recursos de partido político',
 'origem_receita::Recursos de pessoas físicas',
 'origem_receita::Rendimentos de aplicações financeiras']

In [26]:
import networkx as nx
import pandas as pd


def describe_path(G, path):
    rows = []

    for i in range(len(path) - 1):
        source = path[i]
        target = path[i + 1]
        edge = G[source][target]

        rows.append({
            "step": i + 1,
            "source": source,
            "target": target,
            "tipo": edge["tipo"],
            "valor_total": edge["valor_total"],
            "qtd_registros": edge["qtd_registros"],
            "weight": edge["weight"],
        })

    return pd.DataFrame(rows)


def shortest_path_summary(G, source, target):
    try:
        path = nx.shortest_path(
            G,
            source=source,
            target=target,
            weight="weight",
        )

        cost = nx.shortest_path_length(
            G,
            source=source,
            target=target,
            weight="weight",
        )

        path_edges = describe_path(G, path)

        candidatos = [
            node for node in path
            if node.startswith("candidato::")
        ]

        partidos = [
            node for node in path
            if node.startswith("partido::")
        ]

        return {
            "source": source,
            "target": target,
            "cost": cost,
            "path_length": len(path) - 1,
            "path": " -> ".join(path),
            "candidatos": " | ".join(candidatos),
            "partidos": " | ".join(partidos),
            "min_edge_value": path_edges["valor_total"].min(),
            "total_edge_value": path_edges["valor_total"].sum(),
        }

    except nx.NetworkXNoPath:
        return None

In [27]:
resultados = []

for source in origens_receita:
    for target in destinos_cnae:
        result = shortest_path_summary(
            G_cat,
            source=source,
            target=target,
        )

        if result is not None:
            resultados.append(result)

paths_df = pd.DataFrame(resultados)

In [28]:
paths_df["candidato_limpo"] = (
    paths_df["candidatos"]
    .str.replace("candidato::", "", regex=False)
)

In [29]:
comparativo = (
    paths_df
    .dropna(subset=["candidato_limpo"])
    .groupby(["candidato_limpo"], as_index=False)
    .agg(
        menor_custo=("cost", "min"),
        custo_medio=("cost", "mean"),
        qtd_caminhos=("cost", "size"),
    )
    .sort_values("menor_custo")
)

comparativo

,candidato_limpo,menor_custo,custo_medio,qtd_caminhos
1,LUIZ INÁCIO LULA DA SILVA,0.260865,0.325754,195
0,JAIR MESSIAS BOLSONARO,0.262204,0.324637,285


In [30]:
row = paths_df.sort_values("cost").iloc[0]

source = row["source"]
target = row["target"]

path = nx.shortest_path(
    G_cat,
    source=source,
    target=target,
    weight="weight",
)

describe_path(G_cat, path)

,step,source,target,tipo,valor_total,qtd_registros,weight
0,1,origem_receita::Recursos de partido político,fonte_receita::FUNDO ESPECIAL,origem_fonte,1.270427e+08,310,0.050865
1,2,fonte_receita::FUNDO ESPECIAL,partido::PT,fonte_partido,1.251190e+08,262,0.050904
2,3,partido::PT,candidato::LUIZ INÁCIO LULA DA SILVA,partido_candidato,1.355393e+08,533,0.050698
3,4,candidato::LUIZ INÁCIO LULA DA SILVA,"tipo_despesa::Produção de programas de rádio, ...",candidato_tipo_despesa,3.789900e+07,2,0.054199
4,5,"tipo_despesa::Produção de programas de rádio, ...",cnae_fornecedor::Agências de publicidade,tipo_despesa_cnae_fornecedor,3.789900e+07,2,0.054199


In [31]:
import plotly.graph_objects as go
import pandas as pd

In [32]:
def clean_label(node):
    return node.split("::", 1)[1] if "::" in node else node

def plot_sankey_from_edges(edges, top_n=40):
    df = edges.copy()

    df = df.sort_values("valor_total", ascending=False).head(top_n)

    labels = pd.Index(
        pd.concat([df["source"], df["target"]]).unique()
    )

    node_id = {label: i for i, label in enumerate(labels)}

    fig = go.Figure(
        data=[
            go.Sankey(
                node=dict(
                    label=[clean_label(x) for x in labels],
                    pad=15,
                    thickness=15,
                ),
                link=dict(
                    source=df["source"].map(node_id),
                    target=df["target"].map(node_id),
                    value=df["valor_total"],
                    customdata=df["tipo"],
                    hovertemplate=(
                        "Origem: %{source.label}<br>"
                        "Destino: %{target.label}<br>"
                        "Valor: R$ %{value:,.2f}<br>"
                        "Tipo: %{customdata}<extra></extra>"
                    ),
                ),
            )
        ]
    )

    fig.update_layout(
        title_text="Fluxo agregado de receitas e despesas das campanhas",
        font_size=10,
        height=700,
    )

    return fig

In [33]:
fig = plot_sankey_from_edges(edges_cat, top_n=40)
fig.show()

In [34]:
fig = plot_sankey_from_edges(edges_cat, top_n=25)
fig.show()

In [35]:
import plotly.graph_objects as go
import pandas as pd


def clean_label(node):
    return node.split("::", 1)[1] if "::" in node else node


def get_candidate_subgraph_edges(edges, candidate_name):
    candidate_node = f"candidato::{candidate_name}"

    incoming = edges[
        edges["target"].eq(candidate_node)
    ].copy()

    outgoing = edges[
        edges["source"].eq(candidate_node)
    ].copy()

    previous_nodes = incoming["source"].unique()

    upstream = edges[
        edges["target"].isin(previous_nodes)
    ].copy()

    next_nodes = outgoing["target"].unique()

    downstream = edges[
        edges["source"].isin(next_nodes)
    ].copy()

    candidate_edges = pd.concat(
        [upstream, incoming, outgoing, downstream],
        ignore_index=True
    ).drop_duplicates()

    return candidate_edges

In [36]:
def plot_candidate_sankey(edges, candidate_name, top_n=40):
    df = get_candidate_subgraph_edges(edges, candidate_name)

    df = df.sort_values("valor_total", ascending=False).head(top_n)

    labels = pd.Index(
        pd.concat([df["source"], df["target"]]).unique()
    )

    node_id = {label: i for i, label in enumerate(labels)}

    fig = go.Figure(
        data=[
            go.Sankey(
                node=dict(
                    label=[clean_label(x) for x in labels],
                    pad=15,
                    thickness=15,
                ),
                link=dict(
                    source=df["source"].map(node_id),
                    target=df["target"].map(node_id),
                    value=df["valor_total"],
                    customdata=df["tipo"],
                    hovertemplate=(
                        "Origem: %{source.label}<br>"
                        "Destino: %{target.label}<br>"
                        "Valor: R$ %{value:,.2f}<br>"
                        "Tipo: %{customdata}<extra></extra>"
                    ),
                ),
            )
        ]
    )

    fig.update_layout(
        title_text=f"Fluxo agregado da campanha: {candidate_name}",
        font_size=10,
        height=700,
    )

    return fig

In [37]:
fig_lula = plot_candidate_sankey(
    edges_cat,
    "LUIZ INÁCIO LULA DA SILVA",
    top_n=40
)

fig_lula.show()

In [38]:
fig_bolsonaro = plot_candidate_sankey(
    edges_cat,
    "JAIR MESSIAS BOLSONARO",
    top_n=40
)

fig_bolsonaro.show()

In [2]:
def path_cost(G, path):
    cost = 0

    for i in range(len(path) - 1):
        cost += G[path[i]][path[i + 1]]["weight"]

    return cost

def candidate_paths(
    G,
    candidate_name,
):
    candidate_node = f"candidato::{candidate_name}"

    origens = [
        n for n in G.nodes
        if n.startswith("origem_receita::")
    ]

    destinos = [
        n for n in G.nodes
        if n.startswith("cnae_fornecedor::")
    ]

    rows = []

    for origem in origens:
        for destino in destinos:

            try:
                path = nx.shortest_path(
                    G,
                    source=origem,
                    target=destino,
                    weight="weight",
                )

                if candidate_node not in path:
                    continue

                rows.append({
                    "origem": origem,
                    "destino": destino,
                    "candidato": candidate_name,
                    "custo": path_cost(G, path),
                    "caminho": " -> ".join(path),
                    "num_passos": len(path) - 1,
                })

            except nx.NetworkXNoPath:
                pass

    return pd.DataFrame(rows)

In [17]:
paths_bolsonaro = candidate_paths(
    G_cat,
    "JAIR MESSIAS BOLSONARO"
)

paths_bolsonaro.sort_values(
    "custo"
).head(10)

,origem,destino,candidato,custo,caminho,num_passos
164,origem_receita::Recursos de pessoas físicas,"cnae_fornecedor::Portais, provedores de conteú...",JAIR MESSIAS BOLSONARO,0.262204,origem_receita::Recursos de pessoas físicas ->...,5
177,origem_receita::Recursos de pessoas físicas,cnae_fornecedor::Agências de publicidade,JAIR MESSIAS BOLSONARO,0.266591,origem_receita::Recursos de pessoas físicas ->...,5
162,origem_receita::Recursos de pessoas físicas,cnae_fornecedor::Agenciamento de espaços para ...,JAIR MESSIAS BOLSONARO,0.269621,origem_receita::Recursos de pessoas físicas ->...,5
100,origem_receita::Recursos de Financiamento Cole...,"cnae_fornecedor::Portais, provedores de conteú...",JAIR MESSIAS BOLSONARO,0.269948,origem_receita::Recursos de Financiamento Cole...,5
168,origem_receita::Recursos de pessoas físicas,cnae_fornecedor::Atividades de produção cinema...,JAIR MESSIAS BOLSONARO,0.270582,origem_receita::Recursos de pessoas físicas ->...,5
187,origem_receita::Recursos de pessoas físicas,"cnae_fornecedor::Impressão de jornais, livros,...",JAIR MESSIAS BOLSONARO,0.271958,origem_receita::Recursos de pessoas físicas ->...,5
166,origem_receita::Recursos de pessoas físicas,cnae_fornecedor::Atividades de organizações po...,JAIR MESSIAS BOLSONARO,0.272339,origem_receita::Recursos de pessoas físicas ->...,5
200,origem_receita::Recursos de pessoas físicas,cnae_fornecedor::Impressão de material de segu...,JAIR MESSIAS BOLSONARO,0.272805,origem_receita::Recursos de pessoas físicas ->...,5
163,origem_receita::Recursos de pessoas físicas,"cnae_fornecedor::Atividades profissionais, cie...",JAIR MESSIAS BOLSONARO,0.273633,origem_receita::Recursos de pessoas físicas ->...,5
210,origem_receita::Recursos de pessoas físicas,cnae_fornecedor::Criação artística,JAIR MESSIAS BOLSONARO,0.273735,origem_receita::Recursos de pessoas físicas ->...,5


In [18]:
paths_lula = candidate_paths(
    G_cat,
    "LUIZ INÁCIO LULA DA SILVA"
)

paths_lula.sort_values(
    "custo"
).head(10)

,origem,destino,candidato,custo,caminho,num_passos
47,origem_receita::Recursos de partido político,cnae_fornecedor::Agências de publicidade,LUIZ INÁCIO LULA DA SILVA,0.260865,origem_receita::Recursos de partido político -...,5
12,origem_receita::Recursos de partido político,"cnae_fornecedor::Portais, provedores de conteú...",LUIZ INÁCIO LULA DA SILVA,0.261225,origem_receita::Recursos de partido político -...,5
25,origem_receita::Recursos de partido político,cnae_fornecedor::Atividades de produção cinema...,LUIZ INÁCIO LULA DA SILVA,0.264857,origem_receita::Recursos de partido político -...,5
10,origem_receita::Recursos de partido político,cnae_fornecedor::Agenciamento de espaços para ...,LUIZ INÁCIO LULA DA SILVA,0.268643,origem_receita::Recursos de partido político -...,5
21,origem_receita::Recursos de partido político,cnae_fornecedor::Atividades de organizações po...,LUIZ INÁCIO LULA DA SILVA,0.269159,origem_receita::Recursos de partido político -...,5
58,origem_receita::Recursos de partido político,cnae_fornecedor::Impressão de materiais para o...,LUIZ INÁCIO LULA DA SILVA,0.269266,origem_receita::Recursos de partido político -...,5
57,origem_receita::Recursos de partido político,"cnae_fornecedor::Impressão de jornais, livros,...",LUIZ INÁCIO LULA DA SILVA,0.270085,origem_receita::Recursos de partido político -...,5
70,origem_receita::Recursos de partido político,cnae_fornecedor::Impressão de material de segu...,LUIZ INÁCIO LULA DA SILVA,0.270932,origem_receita::Recursos de partido político -...,5
24,origem_receita::Recursos de partido político,"cnae_fornecedor::Artes cênicas, espetáculos e ...",LUIZ INÁCIO LULA DA SILVA,0.272223,origem_receita::Recursos de partido político -...,5
51,origem_receita::Recursos de partido político,cnae_fornecedor::Edição de livros,LUIZ INÁCIO LULA DA SILVA,0.272244,origem_receita::Recursos de partido político -...,5


In [19]:
comparacao = pd.concat(
    [
        paths_lula,
        paths_bolsonaro,
    ],
    ignore_index=True
)

In [21]:
comparacao.groupby(
    "candidato",
    as_index=False
).agg(
    menor_custo=("custo", "min"),
    custo_medio=("custo", "mean"),
    qtd_caminhos=("custo", "count"),
)

,candidato,menor_custo,custo_medio,qtd_caminhos
0,JAIR MESSIAS BOLSONARO,0.262204,0.324637,285
1,LUIZ INÁCIO LULA DA SILVA,0.260865,0.325754,195


In [22]:
despesas_pres.groupby("NM_CANDIDATO")["DS_ORIGEM_DESPESA"].nunique()

NM_CANDIDATO
JAIR MESSIAS BOLSONARO       24
LUIZ INÁCIO LULA DA SILVA    25
Name: DS_ORIGEM_DESPESA, dtype: int64

In [23]:
despesas_pres.groupby("NM_CANDIDATO")["DS_CNAE_FORNECEDOR"].nunique()

NM_CANDIDATO
JAIR MESSIAS BOLSONARO       66
LUIZ INÁCIO LULA DA SILVA    66
Name: DS_CNAE_FORNECEDOR, dtype: int64

In [24]:
receitas_pres.groupby("NM_CANDIDATO")["DS_ORIGEM_RECEITA"].nunique()

NM_CANDIDATO
JAIR MESSIAS BOLSONARO       5
LUIZ INÁCIO LULA DA SILVA    4
Name: DS_ORIGEM_RECEITA, dtype: int64

In [25]:
receitas_pres.groupby("NM_CANDIDATO")["DS_FONTE_RECEITA"].nunique()

NM_CANDIDATO
JAIR MESSIAS BOLSONARO       3
LUIZ INÁCIO LULA DA SILVA    3
Name: DS_FONTE_RECEITA, dtype: int64

In [26]:
paths_bolsonaro.sort_values("custo").head(10)[
    ["origem", "destino", "custo", "caminho"]
]

,origem,destino,custo,caminho
164,origem_receita::Recursos de pessoas físicas,"cnae_fornecedor::Portais, provedores de conteú...",0.262204,origem_receita::Recursos de pessoas físicas ->...
177,origem_receita::Recursos de pessoas físicas,cnae_fornecedor::Agências de publicidade,0.266591,origem_receita::Recursos de pessoas físicas ->...
162,origem_receita::Recursos de pessoas físicas,cnae_fornecedor::Agenciamento de espaços para ...,0.269621,origem_receita::Recursos de pessoas físicas ->...
100,origem_receita::Recursos de Financiamento Cole...,"cnae_fornecedor::Portais, provedores de conteú...",0.269948,origem_receita::Recursos de Financiamento Cole...
168,origem_receita::Recursos de pessoas físicas,cnae_fornecedor::Atividades de produção cinema...,0.270582,origem_receita::Recursos de pessoas físicas ->...
187,origem_receita::Recursos de pessoas físicas,"cnae_fornecedor::Impressão de jornais, livros,...",0.271958,origem_receita::Recursos de pessoas físicas ->...
166,origem_receita::Recursos de pessoas físicas,cnae_fornecedor::Atividades de organizações po...,0.272339,origem_receita::Recursos de pessoas físicas ->...
200,origem_receita::Recursos de pessoas físicas,cnae_fornecedor::Impressão de material de segu...,0.272805,origem_receita::Recursos de pessoas físicas ->...
163,origem_receita::Recursos de pessoas físicas,"cnae_fornecedor::Atividades profissionais, cie...",0.273633,origem_receita::Recursos de pessoas físicas ->...
210,origem_receita::Recursos de pessoas físicas,cnae_fornecedor::Criação artística,0.273735,origem_receita::Recursos de pessoas físicas ->...


In [27]:
paths_lula.sort_values("custo").head(10)[
    ["origem", "destino", "custo", "caminho"]
]

,origem,destino,custo,caminho
47,origem_receita::Recursos de partido político,cnae_fornecedor::Agências de publicidade,0.260865,origem_receita::Recursos de partido político -...
12,origem_receita::Recursos de partido político,"cnae_fornecedor::Portais, provedores de conteú...",0.261225,origem_receita::Recursos de partido político -...
25,origem_receita::Recursos de partido político,cnae_fornecedor::Atividades de produção cinema...,0.264857,origem_receita::Recursos de partido político -...
10,origem_receita::Recursos de partido político,cnae_fornecedor::Agenciamento de espaços para ...,0.268643,origem_receita::Recursos de partido político -...
21,origem_receita::Recursos de partido político,cnae_fornecedor::Atividades de organizações po...,0.269159,origem_receita::Recursos de partido político -...
58,origem_receita::Recursos de partido político,cnae_fornecedor::Impressão de materiais para o...,0.269266,origem_receita::Recursos de partido político -...
57,origem_receita::Recursos de partido político,"cnae_fornecedor::Impressão de jornais, livros,...",0.270085,origem_receita::Recursos de partido político -...
70,origem_receita::Recursos de partido político,cnae_fornecedor::Impressão de material de segu...,0.270932,origem_receita::Recursos de partido político -...
24,origem_receita::Recursos de partido político,"cnae_fornecedor::Artes cênicas, espetáculos e ...",0.272223,origem_receita::Recursos de partido político -...
51,origem_receita::Recursos de partido político,cnae_fornecedor::Edição de livros,0.272244,origem_receita::Recursos de partido político -...
